# Adaptive Shield Weekly Report (MVP)

"
"This notebook automates weekly reporting for **configuration drift** and **integration failure** alerts.

"
"## Pipeline
"
"1. Fetch alerts in lookback window
"
"2. Filter target alert types
"
"3. Enrich with security check details
"
"4. Enrich affected entities
"
"5. Build summary + entities tables
"
"6. Export XLSX + CSV

"
"ServiceNow enrichment is a stub in this MVP.

In [ ]:
# Cell 1: Imports + Configuration Loader
import os
import sys
import logging
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

ROOT_DIR = Path.cwd().resolve()
SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

load_dotenv(ROOT_DIR / ".env")


def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_TS = datetime.now(timezone.utc)
LOOKBACK_DAYS = int(os.getenv("LOOKBACK_DAYS", "3"))
FROM_DATE = (RUN_TS - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")
TO_DATE = RUN_TS.strftime("%Y-%m-%dT%H:%M:%SZ")

config = {
    "as_api_key": os.getenv("AS_API_KEY", "").strip(),
    "as_base_url": os.getenv("AS_BASE_URL", "https://api.adaptive-shield.com").strip(),
    "as_account_ids": [
        account.strip()
        for account in os.getenv("AS_ACCOUNT_IDS", "").split(",")
        if account.strip()
    ],
    "lookback_days": LOOKBACK_DAYS,
    "from_date": FROM_DATE,
    "to_date": TO_DATE,
    "output_root": os.getenv("OUTPUT_ROOT", "output").strip(),
    "export_xlsx": env_bool("EXPORT_XLSX", True),
    "export_csv": env_bool("EXPORT_CSV", True),
    "snow_enabled": env_bool("SNOW_ENABLED", False),
    "rate_limit_per_minute": int(os.getenv("RATE_LIMIT_PER_MINUTE", "90")),
    "request_timeout_seconds": int(os.getenv("REQUEST_TIMEOUT_SECONDS", "30")),
    "max_retries": int(os.getenv("MAX_RETRIES", "3")),
}

if not config["as_api_key"]:
    raise ValueError("AS_API_KEY is required. Add it to .env before running.")

RUN_DAY_DIR = ROOT_DIR / config["output_root"] / RUN_TS.strftime("%Y-%m-%d")
RUN_DAY_DIR.mkdir(parents=True, exist_ok=True)

pipeline_errors = []

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
print("Configuration loaded:")
print({k: v for k, v in config.items() if k != "as_api_key"})
print("Output directory:", RUN_DAY_DIR)


In [ ]:
# Cell 2: API Client Initialization
from as_weekly_report.as_client import AdaptiveShieldClient

client = AdaptiveShieldClient(
    api_key=config["as_api_key"],
    base_url=config["as_base_url"],
    rate_limit_per_minute=config["rate_limit_per_minute"],
    timeout_seconds=config["request_timeout_seconds"],
    max_retries=config["max_retries"],
)

print("AdaptiveShieldClient initialized")


In [ ]:
# Cell 3: Get Accounts
accounts_records = []

if config["as_account_ids"]:
    for account_id in config["as_account_ids"]:
        accounts_records.append({"id": account_id, "name": ""})
else:
    accounts_records = client.get_accounts()

accounts_df = pd.DataFrame(accounts_records)
if not accounts_df.empty:
    accounts_df = accounts_df.rename(columns={"id": "account_id", "name": "account_name"})
    for column in ["account_id", "account_name"]:
        if column not in accounts_df.columns:
            accounts_df[column] = ""
    accounts_df = accounts_df[["account_id", "account_name"]]
else:
    accounts_df = pd.DataFrame(columns=["account_id", "account_name"])

display(accounts_df.head(20))
print("Accounts found:", len(accounts_df))


In [ ]:
# Cell 4: Fetch Alerts
from as_weekly_report.transform import filter_target_alerts

TARGET_ALERT_TYPES = ["configuration_drift", "integration_failure"]

alerts_raw_rows = []
for account_id in accounts_df["account_id"].dropna().astype(str).tolist():
    for alert_type in TARGET_ALERT_TYPES:
        try:
            rows = client.get_alerts(
                account_id=account_id,
                from_date=config["from_date"],
                to_date=config["to_date"],
                alert_type=alert_type,
            )
            for row in rows:
                row.setdefault("account_id", account_id)
            alerts_raw_rows.extend(rows)
        except Exception as exc:
            pipeline_errors.append(
                {
                    "stage": "alerts",
                    "account_id": account_id,
                    "security_check_id": None,
                    "alert_id": None,
                    "message": str(exc),
                }
            )

alerts_raw_df = pd.DataFrame(alerts_raw_rows)
if not alerts_raw_df.empty and "id" in alerts_raw_df.columns:
    alerts_raw_df = alerts_raw_df.drop_duplicates(subset=["account_id", "id"], keep="first")

alerts_filtered_rows = filter_target_alerts(alerts_raw_df.to_dict("records"), include_check_degraded=False)
alerts_filtered_df = pd.DataFrame(alerts_filtered_rows)

display(alerts_filtered_df.head(20))
print("Raw alerts:", len(alerts_raw_df), "| Filtered alerts:", len(alerts_filtered_df))


In [ ]:
# Cell 5: Enrich Security Checks
from as_weekly_report.transform import extract_security_check_id

check_cache = {}
check_rows = []

for alert in alerts_filtered_df.to_dict("records"):
    alert_id = str(alert.get("id") or "")
    account_id = str(alert.get("account_id") or "")
    security_check_id = extract_security_check_id(alert)

    if not account_id or not security_check_id:
        pipeline_errors.append(
            {
                "stage": "security_check",
                "account_id": account_id,
                "security_check_id": security_check_id,
                "alert_id": alert_id,
                "message": "Missing account_id or security_check_id",
            }
        )
        check_rows.append(
            {
                "alert_id": alert_id,
                "account_id": account_id,
                "security_check_id": security_check_id,
            }
        )
        continue

    cache_key = (account_id, security_check_id)
    if cache_key not in check_cache:
        try:
            check_cache[cache_key] = client.get_security_check(account_id, security_check_id)
        except Exception as exc:
            check_cache[cache_key] = {}
            pipeline_errors.append(
                {
                    "stage": "security_check",
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "alert_id": alert_id,
                    "message": str(exc),
                }
            )

    check_data = dict(check_cache[cache_key])
    check_rows.append(
        {
            "alert_id": alert_id,
            "account_id": account_id,
            "security_check_id": security_check_id,
            **check_data,
        }
    )

checks_df = pd.DataFrame(check_rows)
display(checks_df.head(20))
print("Security checks enriched:", len(checks_df))


In [ ]:
# Cell 6: Enrich Affected Entities
from as_weekly_report.transform import build_entities_df, extract_security_check_id

entity_records = []
affected_resolve_log_rows = []
affected_cache = {}

for alert in alerts_filtered_df.to_dict("records"):
    alert_id = str(alert.get("id") or "")
    account_id = str(alert.get("account_id") or "")
    security_check_id = extract_security_check_id(alert)

    if "alert_id" in checks_df.columns:
        check_row = checks_df[checks_df["alert_id"].astype(str) == alert_id]
        check_obj = check_row.iloc[0].to_dict() if not check_row.empty else {}
    else:
        check_obj = {}
    is_global = bool(check_obj.get("is_global"))

    if is_global:
        affected_resolve_log_rows.append(
            {
                "alert_id": alert_id,
                "account_id": account_id,
                "security_check_id": security_check_id,
                "resolve_status": "global",
                "message": "Security check marked as global",
            }
        )
        continue

    cache_key = (account_id, security_check_id)
    entities = []
    if cache_key in affected_cache:
        entities = affected_cache[cache_key]
    else:
        try:
            entities = client.get_affected_entities(account_id, security_check_id)
            affected_cache[cache_key] = entities
        except Exception as exc:
            pipeline_errors.append(
                {
                    "stage": "affected_entities",
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "alert_id": alert_id,
                    "message": str(exc),
                }
            )
            affected_resolve_log_rows.append(
                {
                    "alert_id": alert_id,
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "resolve_status": "unresolved",
                    "message": "Affected API failed",
                }
            )
            continue

    for entity in entities:
        entity_records.append(
            {
                "alert_id": alert_id,
                "account_id": account_id,
                "security_check_id": security_check_id,
                "entity": entity,
            }
        )

    affected_resolve_log_rows.append(
        {
            "alert_id": alert_id,
            "account_id": account_id,
            "security_check_id": security_check_id,
            "resolve_status": "fetched",
            "message": f"Fetched {len(entities)} entities",
        }
    )

entities_df = build_entities_df(entity_records)
affected_resolve_log_df = pd.DataFrame(affected_resolve_log_rows)

display(entities_df.head(20))
display(affected_resolve_log_df.head(20))
print("Entity rows:", len(entities_df))


In [ ]:
# Cell 7: Build Summary Table
from as_weekly_report.transform import build_summary_df

summary_df = build_summary_df(
    alerts=alerts_filtered_df.to_dict("records"),
    checks=checks_df.to_dict("records"),
    entities=entities_df.to_dict("records"),
    accounts=accounts_df.rename(columns={"account_id": "id", "account_name": "name"}).to_dict("records"),
    extracted_at_utc=RUN_TS.strftime("%Y-%m-%dT%H:%M:%SZ"),
)

display(summary_df.head(20))
print("Summary rows:", len(summary_df))


In [ ]:
# Cell 8: ServiceNow Stub
from as_weekly_report.snow_client import fetch_related_tickets, merge_snow_columns

snow_df = fetch_related_tickets(summary_df, config["lookback_days"]) if config["snow_enabled"] else fetch_related_tickets(summary_df, config["lookback_days"])
summary_with_snow_df = merge_snow_columns(summary_df, snow_df)

display(snow_df.head(20))
display(summary_with_snow_df.head(20))


In [ ]:
# Cell 9: Data Quality Checks
from as_weekly_report.transform import SUMMARY_COLUMNS, ENTITY_COLUMNS

quality_rows = []

missing_summary_columns = [col for col in SUMMARY_COLUMNS if col not in summary_with_snow_df.columns]
missing_entity_columns = [col for col in ENTITY_COLUMNS if col not in entities_df.columns]

quality_rows.append(
    {
        "check": "missing_summary_columns",
        "value": len(missing_summary_columns),
        "details": ", ".join(missing_summary_columns),
    }
)
quality_rows.append(
    {
        "check": "missing_entity_columns",
        "value": len(missing_entity_columns),
        "details": ", ".join(missing_entity_columns),
    }
)

if not summary_with_snow_df.empty:
    duplicate_alert_keys = summary_with_snow_df.duplicated(subset=["account_id", "alert_id"]).sum()
else:
    duplicate_alert_keys = 0

quality_rows.append(
    {
        "check": "duplicate_account_alert_keys",
        "value": int(duplicate_alert_keys),
        "details": "subset=[account_id, alert_id]",
    }
)

for column in ["change_datetime", "security_check_name", "impact_level", "current_status"]:
    if column in summary_with_snow_df.columns and not summary_with_snow_df.empty:
        null_ratio = float(summary_with_snow_df[column].isna().mean())
    else:
        null_ratio = 1.0
    quality_rows.append(
        {
            "check": f"null_ratio::{column}",
            "value": round(null_ratio, 4),
            "details": "fraction of null values",
        }
    )

quality_report_df = pd.DataFrame(quality_rows)
errors_df = pd.DataFrame(pipeline_errors)

display(quality_report_df)
display(errors_df.head(20))
print("Pipeline errors:", len(errors_df))


In [ ]:
# Cell 10: Export Files
from as_weekly_report.exporter import export_all

ts = RUN_TS.strftime("%Y%m%d_%H%M%S")
export_paths = export_all(
    summary_df=summary_with_snow_df,
    entities_df=entities_df,
    errors_df=errors_df,
    output_dir=str(RUN_DAY_DIR),
    ts=ts,
    export_xlsx=config["export_xlsx"],
    export_csv=config["export_csv"],
)

print("Export paths:")
for key, value in export_paths.items():
    print(f"- {key}: {value}")
